# Phase 7: QLoRA fine-tuning of the intent parser (GPU)

Trains LoRA adapters on Qwen2.5-1.5B and/or 3B (4-bit base, fp16 compute) from `finetune/data/sft_train.jsonl`,
then parses every evaluation task with the trained adapter (zero-shot, same system prompt as training).
Scoring runs on the laptop with `eval/score_parsers.py`.

**Before running:** the SFT files must be in the repo (`finetune/data/sft_train.jsonl`, `sft_val.jsonl`, built with
`finetune/data_gen/build_sft.py`), or attached as a Kaggle dataset (set `SFT_DIR`).

**Settings:** GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.
Set `SMOKE = True` first: 50 steps per model, to check memory and that the loss falls (README Phase 7, task 3).

Outputs to download: `ft_<model>.zip` (adapter, training log, run info) and `parse_ft_<model>.zip` (parses + `system.txt`).

In [9]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
SFT_DIR = "finetune/data/v2"   # data v2: v1 + review fixes (docs/open_issues.md 2026-10-31); None = finetune/data/
SMOKE = True               # True: 50 steps per model (check); False: full training
# (config file, system name used on the laptop, base model for parsing)
RUNS = [
    ("finetune/configs/qwen25_1_5b.yaml", "intent_ft_1_5b", "Qwen/Qwen2.5-1.5B-Instruct"),
    ("finetune/configs/qwen25_3b.yaml",   "intent_ft_3b",   "Qwen/Qwen2.5-3B-Instruct"),
]
PARSE_AFTER_TRAINING = True    # parse all evaluation tasks with each adapter (skipped in SMOKE mode)

In [10]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, json, time
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
if SFT_DIR:
    os.makedirs("finetune/data", exist_ok=True)
    for f in ("sft_train.jsonl", "sft_val.jsonl", "sft_stats.json"):
        if os.path.exists(f"{SFT_DIR}/{f}"):
            shutil.copy(f"{SFT_DIR}/{f}", f"finetune/data/{f}")
NEEDED = ["finetune/train_qlora.py", "eval/parse_contracts.py", "src/igt/agents/intent_parser.py",
          "finetune/data/sft_train.jsonl", "finetune/data/sft_val.jsonl"] + [r[0] for r in RUNS]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them, or set SFT_DIR): {missing}"
n_train = sum(1 for _ in open("finetune/data/sft_train.jsonl"))
n_val = sum(1 for _ in open("finetune/data/sft_val.jsonl"))
print(f"SFT data: {n_train} train, {n_val} val")

no GITHUB_TOKEN secret; cloning without a token (public repo)


Cloning into '/kaggle/working/repo'...


commit 6a79a3690433907d19a5d92242817ee1898a05d2
SFT data: 3614 train, 796 val


In [11]:
# ---------------- Libraries ----------------
subprocess.run(["pip", "install", "-q", "-r", "finetune/requirements-train.txt"], check=True)
import torch, transformers, peft
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available()

GPU: Tesla T4
torch 2.11.0+cu128 | transformers 5.16.1 | peft 0.21.2


In [ ]:
# ---------------- Train (and parse) each model; one subprocess per step frees GPU memory ----------------
def sh(cmd):
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-2500:]); print(p.stderr[-2500:])
    assert p.returncode == 0, cmd
    return (time.time() - t0) / 60

for config, system, base in RUNS:
    tag = os.path.basename(config).replace(".yaml", "")
    out = f"finetune/runs/{tag}" + ("_smoke" if SMOKE else "")
    sets = [f"output_dir={out}"] + (["max_steps=50", "eval_steps=25", "logging_steps=5"] if SMOKE else [])
    print(f"===== train {system} ({config}) -> {out}")
    mins = sh(["python", "finetune/train_qlora.py", "--config", config, "--set", *sets])
    info = json.load(open(f"{out}/run_info.json"))
    print(f"trained in {mins:.1f} min | loss {info['first_logged_loss']} -> {info['last_logged_loss']} | eval {info['final_eval'].get('eval_loss')}")
    open(f"{out}/git_commit.txt", "w").write(COMMIT)
    print("saved:", shutil.make_archive(f"/kaggle/working/ft_{tag}" + ("_smoke" if SMOKE else ""), "zip", out))
    if PARSE_AFTER_TRAINING and not SMOKE:
        pout = f"/kaggle/working/parse_ft_{tag}"
        print(f"===== parse with {system}")
        mins = sh(["python", "eval/parse_contracts.py", "--model", base, "--load-in-4bit",
                   "--adapter", f"{out}/adapter", "--out", pout, "--batch", "4"])
        open(f"{pout}/system.txt", "w").write(system)
        open(f"{pout}/git_commit.txt", "w").write(COMMIT)
        print(f"parsed in {mins:.1f} min; saved:", shutil.make_archive(pout, "zip", pout))

===== train intent_ft_1_5b (finetune/configs/qwen25_1_5b.yaml) -> finetune/runs/qwen25_1_5b_smoke


In [ ]:
# ---------------- Clean up (keep only the zips and outputs) ----------------
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
print(sorted(f for f in os.listdir("/kaggle/working") if f.endswith(".zip")))